# DIZA Imagine · WAN 2.2 I2V · Kaggle Backend

Backend terpisah untuk menerima image + prompt dari web DIZA dan menghasilkan MP4 dengan Wan2.2-TI2V-5B low-VRAM. Prompt positif diteruskan apa adanya.

In [ ]:
import os, subprocess, sys
print('=== GPU ===')
subprocess.run(['nvidia-smi'])
print('=== INSTALL ===')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/modelscope/DiffSynth-Studio.git', 'fastapi', 'uvicorn[standard]', 'python-multipart', 'requests'])
print('INSTALL_OK')

In [ ]:
from pathlib import Path
import requests, py_compile
BACKEND_URL='https://raw.githubusercontent.com/fatonyist92-jpg/Diza-Imagine-Wan/main/kaggle_wan22_backend.py'
BACKEND_PATH=Path('/kaggle/working/kaggle_wan22_backend.py')
r=requests.get(BACKEND_URL, timeout=30)
r.raise_for_status()
BACKEND_PATH.write_bytes(r.content)
py_compile.compile(str(BACKEND_PATH), doraise=True)
print('BACKEND_OK', BACKEND_PATH)

In [ ]:
import subprocess, sys, time, requests, os
os.environ['WAN_MODEL_ID']='Wan-AI/Wan2.2-TI2V-5B'
api_log=open('/kaggle/working/diza_wan_api.log','w')
API_PROC=subprocess.Popen([sys.executable,'-m','uvicorn','kaggle_wan22_backend:app','--host','0.0.0.0','--port','8000'], cwd='/kaggle/working', stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        h=requests.get('http://127.0.0.1:8000/health', timeout=2)
        if h.ok:
            print('API_OK', h.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError(open('/kaggle/working/diza_wan_api.log').read()[-5000:])

In [ ]:
import os, stat, subprocess, requests, re, select, time
CF='/kaggle/working/cloudflared'
if not os.path.exists(CF):
    data=requests.get('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', timeout=120).content
    open(CF,'wb').write(data)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
TUNNEL_PROC=subprocess.Popen([CF,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
PUBLIC_URL=None
deadline=time.time()+90
while time.time()<deadline and PUBLIC_URL is None:
    ready,_,_=select.select([TUNNEL_PROC.stdout],[],[],1)
    if not ready:
        continue
    line=TUNNEL_PROC.stdout.readline()
    print(line.rstrip())
    m=re.search(r'https://[a-zA-Z0-9-]+\\.trycloudflare\\.com', line)
    if m:
        PUBLIC_URL=m.group(0)
if not PUBLIC_URL:
    raise RuntimeError('Cloudflare tunnel URL tidak ditemukan')
print('\nDIZA_KAGGLE_URL =', PUBLIC_URL)

In [ ]:
import requests, json
h=requests.get(PUBLIC_URL+'/health', timeout=30)
h.raise_for_status()
print(json.dumps(h.json(), indent=2))
print('\nBACKEND_READY:', PUBLIC_URL)

In [ ]:
# Keep this cell running while DIZA Imagine uses the Kaggle GPU backend.
import time
print('DIZA WAN backend aktif. Stop notebook untuk mematikannya.')
while True:
    time.sleep(60)